# Proteomics, Post-translational Modifications, andIntegrative Analyses Reveal MolecularHeterogeneity within Medulloblastoma Subgroups

## PROTEOMICS DATASET RE-ANALYSIS


### **Abstract**
There is a pressing need to identify therapeutic targets in tumors with low mutation rates such as the malignant pediatric brain tumor medulloblastoma. To address this challenge, we quantitatively profiled global proteomes and phospho-proteomes of 45 medulloblastoma samples. Integrated analyses revealed that tumors with similar RNA expression vary extensively at the post-transcriptional and post-translational levels. We identified distinct pathways associated with two subsets of SHH tumors, and found post-translational modifications of MYC that are associated with poor outcomes in Group 3 tumors. We found kinases associated with subtypes and showed that inhibiting PRKDC sensitizes MYC-driven cells to radiation. Our study shows that proteomics enables a more comprehensive, functional readout, providing a foundation for future therapeutic strategies.

![experimental_design](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6372116/bin/nihms-1503512-f0001.jpg)
**Figure 1:** Summary of data types included in this study, depth of proteomic data types, and cohort composition.

Consensus clustering, principal component analysis (PCA), and t-distributed stochastic neighbor embedding (t-SNE) carried out separately on each data type predominantly revealed the known subgroups: Group 3, Group 4, and SHH (Figures ​(Figures2A,2A, S1-S3). The proteomic data also identified very stable subsets of two known subgroups. Here, we refer to Group 3 clusters as Group 3a (G3a) and Group 3b (G3b), and SHH clusters as SHHa and SHHb.

![subgroups](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6372116/bin/nihms-1503512-f0002.jpg)
**Figure 2:** Comparison of clustering results.

REF: https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6372116/

### Objectives
Use **CKG's analytics core** to:

1. Re-analyze the proteomics dataset: 

    - Preprocessing
    - Stratification looking at covariates: Age, Gender, Metastatic status and histology
    - Differential regulation including covariates
    - Functional enrichment

2. Identify patients subgroups 

    - Integration of all omics datasets using Similarity Network Fusion (REF: https://www.nature.com/articles/nmeth.2810).
    - Idenfity clinical and molecular differences between the identifierd subgroups

3. Investigate correlating PTMs on relevant protein complexes

# Loading the Data

In this analysis we use:

- Clinical metadata
- RNA-sequencing 
- Proteomics
- Phosphoproteomics
- Phosphotyrosine proteomics
- Acetylomics

This datasets have been previously formated (samples x proteins) and standardized to UniProt identifiers (data available in */assets*) (see notebook Meduloblastoma Data Processing.ipynb).

In [ ]:
import pandas as pd
import numpy as np

from analytics_core.analytics import analytics
from analytics_core.viz import viz

from graphdb_builder import mapping
from graphdb_connector import connector

from plotly.offline import init_notebook_mode, iplot

%matplotlib inline
init_notebook_mode(connected=True)

In [ ]:
clinical_data = pd.read_excel('assets/samples.xlsx', header=0)
rnaseq_data = pd.read_csv("assets/rnaseq.tsv", sep='\t', header=0)
proteomics_data = pd.read_csv("assets/proteomics.tsv", sep='\t', header=0)
phosphoSTY_data = pd.read_csv("assets/phosphoSTY.tsv", sep='\t', header=0)
phosphoY_data = pd.read_csv("assets/phosphoY.tsv", sep='\t', header=0)
acetylomics_data = pd.read_csv("assets/acetylomics.tsv", sep='\t', header=0)

# 1. Re-analyze the proteomics dataset

In [ ]:
proteomics_data.head()

## Preprocessing:

0. Remove WNT samples (low number of samples)
1. Filter proteins with more than 80% missing values in all groups
2. Normalize data using median-centered normalization
3. Impute missing values using a mixed model: K-Nearest Neighbors algorithm (KNN) if at least 60% valid values, otherwise Minimum Probability (MinProb) drawing random values from a down-shifted Gaussian distribution
4. Include Protein name in columns using CKG's mapping functionality
5. Include relevant clinical features: Age, Gender, Metastatic status, histology


#### 0. Remove WNT samples

In [ ]:
proteomics_data = proteomics_data[proteomics_data['group'] != 'WNT']

#### 1. Filter proteins with more than 80% missing values in all groups

In [ ]:
valid_proteins = analytics.extract_percentage_missing(proteomics_data, missing_max=0.2, drop_cols=['index'], group='group', how='all')
proteomics_data = proteomics_data[valid_proteins + ['index', 'group']]

#### 2. Normalize data using median-centered normalization

In [ ]:
proteomics_data = analytics.normalize_data(proteomics_data, method='median', normalize='samples')


#### 3. Impute missing values using a mixed model

In [ ]:
proteomics_data = analytics.imputation_mixed_norm_KNN(proteomics_data, index_cols=['index','group']).reset_index()
proteomics_data = proteomics_data.rename(columns={'index':'sample'})

#### 4. Include Protein name in columns

In [ ]:
m = mapping.getMappingFromDatabase(proteomics_data.columns, node="Protein", attribute_from='id', attribute_to='name')
columns = [m[c]+"~"+c if c in m and m[c] is not None else c for c in proteomics_data.columns]
proteomics_data.columns = columns

In [ ]:
proteomics_data.head()

#### 5. Include relevant clinical features

In [ ]:
    proteomics_data = proteomics_data.set_index(['sample', 'group']).join(clinical_data.set_index(['sample', 'group'])).reset_index()
    proteomics_data = proteomics_data.drop('Proteome-based ', axis=1)
    proteomics_data.loc[proteomics_data['Metastatic  Status'].isna(), 'Metastatic  Status'] = 'M3'

In [ ]:
proteomics_data.head()

## Sample Stratification

We use Principal Component Analysis to visualize whether the groups cluster together.

In [ ]:
pca, args = analytics.run_pca(proteomics_data, drop_cols=['Gender', 'Metastatic  Status', 'Age At Diagnosis (years)', 'Histology'], group='group', annotation_cols=['sample'], components=2)



In [ ]:
args.update({'title':"PCA Medulloblastoma groups", 'loadings':10, 'factor': 400})
figure = viz.get_pca_plot(pca, identifier='pca_plot', args=args)
iplot(figure.figure)
viz.save_DASH_plot(figure, name="PCA_Medulloblastoma_groups", plot_format='png', directory='.')

We can also color based on possible covariates: Age, Gender, Metastatic status and histology

In [ ]:
pca, args = analytics.run_pca(proteomics_data, drop_cols=['Proteome-based', 'Gender', 'group', 'Metastatic  Status', 'group', 'Gender', 'Histology'], group='Age At Diagnosis (years)', annotation_cols=['sample'], components=2)



In [ ]:
args.update({'title':"PCA Medulloblastoma by Age", 'loadings':10, 'factor': 400})
figure = viz.get_pca_plot(pca, identifier='pca_plot', args=args)
iplot(figure.figure)
viz.save_DASH_plot(figure, name="PCA_Medulloblastoma_age", plot_format='png', directory='.')

In [ ]:
pca, args = analytics.run_pca(proteomics_data, drop_cols=['Proteome-based', 'Age At Diagnosis (years)', 'group', 'Metastatic  Status', 'Histology'], group='Gender', annotation_cols=['sample'], components=2)



In [ ]:
pca, args = analytics.run_pca(proteomics_data, drop_cols=['Proteome-based', 'Gender', 'group','Age At Diagnosis (years)' , 'Gender', 'Histology'], group='Metastatic  Status', annotation_cols=['sample'], components=2)



In [ ]:
args.update({'title':"PCA Medulloblastoma by Metastatic Status", 'loadings':10, 'factor': 400})
figure = viz.get_pca_plot(pca, identifier='pca_plot', args=args)
iplot(figure.figure)
viz.save_DASH_plot(figure, name="PCA_Medulloblastoma_metastatics", plot_format='png', directory='.')

In [ ]:
pca, args = analytics.run_pca(proteomics_data, drop_cols=['Proteome-based', 'Gender', 'group', 'Metastatic  Status', 'group', 'Gender', 'Age At Diagnosis (years)'], group='Histology', annotation_cols=['sample'], components=2)



In [ ]:
args.update({'title':"PCA Medulloblastoma by Histology", 'loadings':10, 'factor': 400})
figure = viz.get_pca_plot(pca, identifier='pca_plot', args=args)
iplot(figure.figure)
viz.save_DASH_plot(figure, name="PCA_Medulloblastoma_histology", plot_format='png', directory='.')

#### single-sample Gene Set Enrichment Analysis

We can also stratify looking at biological processes (GO terms) common to the studied cohort. We use single-sample GSEA (REF:https://www.ncbi.nlm.nih.gov/pmc/articles/PMC2783335/) to project each sample onto a 2D space of normalized gene set enrichment scores.

We extract the Gene Ontology annotation from CKG's graph database using the connector module.

In [ ]:
go_terms_query = "MATCH (p:Protein)-[]-(bp:Biological_process) WHERE (p.name+'~'+p.id) IN {} RETURN DISTINCT (p.name+'~'+p.id) AS identifier,bp.name AS annotation"
go_terms_query = go_terms_query.format(proteomics_data.columns.tolist())
annotation = connector.run_query(go_terms_query)

In [ ]:
annotation.head()

In [ ]:
ssgsea_data = analytics.run_ssgsea(data=proteomics_data.drop(['Metastatic  Status', 'Gender', 'Histology', 'Age At Diagnosis (years)'], axis=1), annotation=annotation, annotation_col='annotation', 
                     identifier_col='identifier', set_index=['group', 'sample'], outdir=None,
                     min_size=10, scale=False, permutations=0)

In [ ]:
pca, args = analytics.run_pca(ssgsea_data['nes'], drop_cols=[ ], group='group', annotation_cols=['sample'], components=2)

In [ ]:
args.update({'title':"Functional PCA Medulloblastoma groups", 'loadings':25, 'factor': 15})
figure = viz.get_pca_plot(pca, identifier='pca_plot', args=args)
iplot(figure.figure)
viz.save_DASH_plot(figure, name="Functional_PCA_Medulloblastoma_groups", plot_format='png', directory='.')

## Differential Regulation

We perform ANCOVA analysis including covariates: Age, Gender, Metastatic status and Histology.

In [ ]:
pro_anova_result = analytics.run_ancova(proteomics_data, covariates=['Age At Diagnosis (years)', 'Gender', 'Histology', 'Metastatic  Status'], drop_cols=['sample'], subject=None, permutations=0)

In [ ]:
pro_anova_result.shape

In [ ]:
anova_significant = pro_anova_result[pro_anova_result.rejected & (pro_anova_result['posthoc padj'] <= 0.01)]

In [ ]:
anova_significant['identifier'].unique().shape

In [ ]:
volcano_plot = viz.run_volcano(pro_anova_result, identifier='proteomics_volcanos', 
            args={'alpha':0.01, 'fc':2, 'num_annotations': 80,
                  'colorscale':'Blues', 'showscale': False, 
                  'marker_size':8, 'x_title':'log2FC', 'y_title':'-log10(pvalue)'})
i = 0
for plot in volcano_plot:
    iplot(plot.figure)
    viz.save_DASH_plot(plot, name="volcano_plot_"+str(i), plot_format='png', directory='.')
    i += 1

## Functional Enrichment Analysis

In [ ]:
enrichment_results = analytics.run_up_down_regulation_enrichment(pro_anova_result, annotation, identifier='identifier', groups=['group1', 'group2'], annotation_col='annotation', reject_col='rejected', group_col='group', method='fisher', correction='fdr_bh', alpha=0.01, lfc_cutoff=1)

In [ ]:
figures = viz.get_enrichment_plots(enrichment_results, identifier='enrichment', args={'width':2200})
i = 0
for fig in figures:
    iplot(fig.figure)
    viz.save_DASH_plot(fig, name="enrichment_"+str(i), plot_format='png', directory='.')
    i += 1

# References

1. https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6372116/